# Phase 8 — Testing & Optimization

This notebook validates the Movie Recommendation System after Phase 7D.

It checks dataset integrity, content-based recommendations, collaborative filtering with SVD, hybrid recommendations, UI code quality, and performance/optimization points.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

MOVIES_FILE = ROOT / 'data' / 'movies.csv'
RATINGS_FILE = ROOT / 'data' / 'ratings.csv'

movies = pd.read_csv(MOVIES_FILE)
ratings = pd.read_csv(RATINGS_FILE)

print('Movies:', movies.shape)
print('Ratings:', ratings.shape)

In [ ]:
# 1. Dataset integrity
assert {'movieId', 'title', 'genres'}.issubset(movies.columns)
assert {'userId', 'movieId', 'rating', 'timestamp'}.issubset(ratings.columns)
assert movies['movieId'].is_unique
assert movies[['movieId', 'title', 'genres']].notna().all().all()
assert ratings[['userId', 'movieId', 'rating']].notna().all().all()

print('PASS: Dataset integrity tests')
print('Users:', ratings['userId'].nunique())
print('Movies:', movies['movieId'].nunique())
print('Ratings:', len(ratings))
print('Average rating:', round(ratings['rating'].mean(), 3))

In [ ]:
# 2. Content-based test
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

m = movies.copy()
m['genres'] = m['genres'].fillna('Unknown').astype(str).str.replace('|', ' ', regex=False)

start = time.perf_counter()
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(m['genres'])
build_time = time.perf_counter() - start

matches = m[m['title'].str.contains('Toy Story', case=False, na=False)]
assert len(matches) > 0

idx = matches.index[0]
scores = cosine_similarity(tfidf[idx], tfidf).flatten()
scores[idx] = -1
top = np.argsort(scores)[::-1][:10]

content_result = m.iloc[top][['movieId', 'title', 'genres']].copy()
content_result['similarity'] = scores[top]

assert len(content_result) == 10
assert content_result['movieId'].is_unique
assert content_result['similarity'].between(0, 1).all()

print(f'PASS: Content-based tests | build time={build_time:.3f}s')
display(content_result)

In [ ]:
# 3. Collaborative filtering / SVD test
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(float(ratings['rating'].min()), float(ratings['rating'].max())))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
start = time.perf_counter()
svd.fit(trainset)
train_time = time.perf_counter() - start

predictions = svd.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae = accuracy.mae(predictions, verbose=False)

assert np.isfinite(rmse)
assert np.isfinite(mae)

print(f'PASS: SVD tests | training={train_time:.3f}s | RMSE={rmse:.4f} | MAE={mae:.4f}')

In [ ]:
# 4. Recommendation-output test for one user
user_id = int(ratings['userId'].iloc[0])
rated = set(ratings.loc[ratings['userId'] == user_id, 'movieId'].astype(int))
candidates = movies[~movies['movieId'].isin(rated)].copy().head(1000)

start = time.perf_counter()
candidates['predicted_rating'] = [svd.predict(user_id, int(mid)).est for mid in candidates['movieId']]
prediction_time = time.perf_counter() - start

result = candidates.sort_values('predicted_rating', ascending=False).head(10)
            
assert len(result) == 10
assert result['movieId'].is_unique
assert not result['movieId'].isin(rated).any()

print(f'PASS: Recommendation output tests | 1000 candidates={prediction_time:.3f}s')
display(result[['movieId', 'title', 'genres', 'predicted_rating']])

## Phase 8 optimization conclusions

- Cache the SVD model with `@st.cache_resource`.
- Cache CSV loading with `@st.cache_data`.
- Do not calculate a complete content similarity matrix on every Streamlit rerun.
- Exclude already-rated movies before ranking.
- Keep model training separate from the UI rendering code.
- Validate recommendation count, uniqueness, and score ranges before display.